In [1]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.1:latest", temperature=0.1, base_url="http://localhost:11434")

In [2]:
print(llm.invoke("Write a 4 line poem of my love for samosa").content)

Here is a 4-line poem about your love for samosas:

My heart beats fast, my taste buds sing
When I take a bite, my love for you takes wing
Samosa's crispy shell and spicy fill
Make my love for you forever stand still.


In [3]:
print(llm.invoke("write email requesting refund for electronic item").content)

Here is an example email you could send to request a refund for an electronic item:

Subject: Request for Refund for [Item Name]

Dear [Customer Service/Return Department],

I am writing to request a refund for the [item name] that I purchased from your company on [date of purchase]. The order number for this transaction is [order number].

Unfortunately, the product has not performed as expected and I have been experiencing issues with [specific issue, e.g. "the screen is cracked" or "it does not turn on"]. Despite my best efforts to troubleshoot the problem, I have been unable to resolve the issue.

I would like to request a full refund for the item, including any shipping costs that were incurred during the purchase process. I have attached a copy of my receipt and a photo of the defective product to this email for your reference.

My order details are as follows:

* Order number: [order number]
* Item name: [item name]
* Date of purchase: [date of purchase]

I would appreciate it i

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [5]:
from few_shots import few_shots
print(len(few_shots), "examples loaded")

14 examples loaded


In [6]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
)
e = embeddings.embed_query("What is your refund policy?")

/home/sunbeam/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4253.52it/s]


In [7]:
len(e)

384

In [8]:
e[:5]

[-0.07025407999753952,
 0.048484064638614655,
 0.048615969717502594,
 0.018366476520895958,
 0.05777846276760101]

In [9]:
vectordb = Chroma.from_texts(
    texts=[ex["Question"] for ex in few_shots],
    embedding=embeddings,
    metadatas=few_shots,
    collection_name="few_shots_nb",
)
retriever = vectordb.as_retriever(search_kwargs={"k": 2})

In [10]:
rdocs = retriever.invoke("how many adidas shirts are in stock?")
for d in rdocs:
    print(d.metadata["Question"], "->", d.metadata["SQLQuery"])

How many total t-shirts are left in stock? -> SELECT SUM(stock_quantity) FROM t_shirts
Which t-shirts are out of stock? -> SELECT brand, color, size FROM t_shirts WHERE stock_quantity = 0
